# User-Based Collaborative Filtering: Taste Through Other People

Popularity gave every user the same list. Content-based tailored the list to genre but could not outpredict mass popularity. This stage tries a completely different source of signal: no item features at all. Instead of describing movies, it finds the users whose taste is most similar to the target user's and recommends what those people rated highly. Collaborative filtering works with nothing but the rating matrix, and it is the family nearly every modern recommender builds on.

## The collaborative filtering family

Collaborative filtering is filtering and evaluating items using the opinions of other people. Content-based recommendation needed a description of every item; collaborative filtering needs no item content whatsoever. The opinions can be whatever a user expresses about an item, and the mechanics only depend on the shape of that signal.

### Ratings and their forms

A rating is an expression of opinion about a user-item pair. Ratings come in several forms. A scalar rating is a point on a continuous or numeric scale, the classic one-to-five stars. An ordinal rating is a rank ordering, such as most-liked to least-liked. A binary rating says the user likes or dislikes an item. A unary rating only says the user engaged with the item at all, like a purchase or a page view, with no direction attached.

Ratings also come from two kinds of feedback. Explicit feedback is given through an interface built to collect opinions: a star click, a like, a thumbs-up. Implicit feedback is observed behavior: a click, a view, time spent, a purchase. The trade-off is the same one seen in the content-based stage. Implicit feedback produces far more signal but with uncertainty about whether the user actually liked the item; explicit feedback is unambiguous but sparse because few users rate most of what they consume. A collaborative filtering system can be built on explicit, implicit, or both.

### How the family evolved

Early collaborative filtering was pull-active: the user had to actively retrieve the opinions of others, as in Usenet discussions where a reader decides whose review to trust. That requires knowing in advance whose opinion is worth trusting. Next came push-active systems, where a user selects a set of trusted users, a hand-picked neighborhood, whose opinions are pushed to them. That requires knowing what content you want, the same burden content-based filtering put on the user. Automated collaborative filtering removed both burdens: the system holds a database of historical opinions, matches the user's own opinions against that database to find similar users, and derives recommendations from them. The user expresses no trust list and no content preference; the ratings speak for themselves.

### What the system does, and the domain it operates in

Functionally, a collaborative filtering system does three things. It can recommend a list of items for a user. It can predict the rating a user would give a specific item. And it can do constrained recommendation: recommendations restricted to a set, such as news from the last day. The three are served by the same scoring machinery.

Three properties of the domain shape the design. Data distribution matters: how many ratings exist per user and per item, and how sparse the matrix is. The underlying meaning matters: what a rating means in this domain, and whether all users use the scale the same way. Data persistence matters: whether opinions live in a long-lived database or stream in briefly, as in news recommendation where yesterday's signal is worthless. MovieLens is a persistent, mostly complete distribution with a shared explicit scale, so it is the classic testbed.

### How the algorithms are organized

The family splits by whether the model is probabilistic. Non-probabilistic approaches include nearest neighbor, which is user-based or item-based, plus graph-based methods, neural network models, and rule mining. Non-probabilistic dimensionality reduction covers SVD, PCA, and factor analysis. Probabilistic approaches build Bayesian networks over the ratings. This stage implements the classic member: user-based nearest neighbor. The rest of the family will appear in later stages of the lab, starting with item-based collaborative filtering.

### User-based nearest neighbor in detail

User-based collaborative filtering is the transpose of the k-NN classifier from the content-based stage. Instead of matching a user to item profiles, it matches a user to other users. Each user is a vector over the items they rated, and the similarity between two users is the Pearson correlation over the items both rated. To score an item for a user, take the ratings the user's nearest neighbors gave that item and weight them by similarity, offset around the user's own mean rating. No movie feature ever enters the computation: the only input is who rated what.

### The two known weaknesses

The classic user-based algorithm carries two documented weaknesses, both visible in this dataset. First, sparse rating data: the average pair of users shares very few co-rated items, and a correlation computed over a handful of items is almost meaningless. Second, the algorithm fails to incorporate population-wide agreement about an item: two users who both enjoy a universally loved blockbuster look similar, even though that agreement says nothing about them specifically. Some algorithms counter the second by weighting items inversely to their popularity when computing user correlation, a variant this stage implements and tests.

### Reference
The collaborative filtering theory in this notebook follows: Schafer, J.B., Frankowski, D., Herlocker, J., Sen, S. (2007). Collaborative Filtering Recommender Systems. In: The Adaptive Web. Lecture Notes in Computer Science, vol 4321. Springer, Berlin, Heidelberg, pp. 291-324. https://doi.org/10.1007/978-3-540-72079-9_9

## Setup

Same loader, same time-based split, same evaluation as the previous two stages, so every number is comparable.

In [1]:
import sys
from pathlib import Path

# set the ROOT directory
ROOT = Path.cwd()
while not (ROOT / "recommendation_lab").is_dir():
    ROOT = ROOT.parent
    if ROOT == ROOT.parent:
        raise RuntimeError("could not locate recommendation_lab package")
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from recommendation_lab.data.loader import load_ml_1m
from recommendation_lab.data.split import time_base_split
from recommendation_lab.evaluation.evaluate import evaluate_predictions, evaluate_ranking
from recommendation_lab.recommenders.content_based import ContentBasedRecommender
from recommendation_lab.recommenders.popularity import PopularityRecommender
from recommendation_lab.recommenders.user_based_cf import UserBasedRecommender

# load dataset
data = load_ml_1m()
ratings, movies = data["ratings"], data["movies"]

# split data
split = time_base_split(ratings)
train, test = split.train, split.test
pd.set_option("display.max_colwidth", 80)
print(f"train: {len(train):,} ratings | test: {len(test):,} ratings")

Dataset 'ml-1m' already present at /Users/kayceejenz/Documents/experiment/recommendation-system-lab/data/ml-1m
train: 797,758 ratings | test: 202,397 ratings


## The signal: who rated what

The only input to user-based collaborative filtering is the user-item matrix: 6,040 rows, one per user, one column per movie, and a rating in the cells where the user rated the movie. There are no genres, no titles, no descriptions. Two users are similar when the cells they have both filled agree.

In [2]:
# the user-item matrix as a sparse array
from scipy import sparse

user_ids = train["user_id"].unique()
item_ids = train["movie_id"].unique()

u_idx = pd.Series(np.arange(len(user_ids)), index=user_ids)
i_idx = pd.Series(np.arange(len(item_ids)), index=item_ids)

rows = u_idx[train["user_id"]].to_numpy()
cols = i_idx[train["movie_id"]].to_numpy()
vals = train["rating"].to_numpy()
R = sparse.csr_matrix((vals, (rows, cols)), shape=(len(user_ids), len(item_ids)))

print(f"user-item matrix: {R.shape[0]:,} users x {R.shape[1]:,} movies")
print(f"ratings: {R.nnz:,}  density: {100 * R.nnz / (R.shape[0] * R.shape[1]):.4f}%")
print(f"ratings per user: mean {np.asarray(R.getnnz(axis=1)).mean():.1f} | median {np.median(np.asarray(R.getnnz(axis=1))):.0f}")
print(f"ratings per movie: mean {np.asarray(R.getnnz(axis=0)).mean():.1f} | median {np.median(np.asarray(R.getnnz(axis=0))):.0f}")

user-item matrix: 6,040 users x 3,667 movies
ratings: 797,758  density: 3.6018%
ratings per user: mean 132.1 | median 76
ratings per movie: mean 217.6 | median 91


**Findings:**
The matrix is 3.60% filled: 132 ratings per user on average across the 3,667 movies rated in train. Each user's vector is almost entirely empty, and two random users share far fewer than 132 movies. Similarity computed over what they share is the whole game, and the sparsity is the game's central difficulty.

## Pearson similarity over co-rated items

Two users are similar when their ratings move together on the items they have both seen. The Pearson correlation does exactly this: for user a and user b, subtract each user's mean, multiply the centered ratings item by item, and normalize by the two standard deviations.

    sim(a, b) = sum_i (r_ai - mean_a)(r_bi - mean_b) / (std_a * std_b)

Restrict the sum to the items both users rated, and the matrix product accumulates co-rated contributions in one pass. The recommender builds this user-user matrix at fit time. User 1's nearest neighbors show what the matrix finds.

In [3]:
ubcf = UserBasedRecommender(k_neighbors=5, min_co_ratings=5).fit(train)

u = int(np.where(ubcf.user_ids == 1)[0][0])
neighbors = ubcf._neighbor_idx[u]
sims = ubcf._neighbor_sim[u]

print(f"user 1 mean rating: {ubcf.user_mean[u]:.2f}")
print("nearest neighbors (user id, Pearson sim, co-rated):")
for v, s in zip(neighbors, sims):
    if s > 0:
        v_row = int(v)
        common = (R[u] != 0).multiply(R[v_row] != 0).nnz
        print(f"  user {ubcf.user_ids[v_row]:6d}  sim {s:+.3f}  co-rated {common:3d}")

user 1 mean rating: 4.19
nearest neighbors (user id, Pearson sim, co-rated):
  user    933  sim +0.986  co-rated   5
  user   3130  sim +0.920  co-rated   5
  user   5910  sim +0.913  co-rated   9
  user   5541  sim +0.903  co-rated   5
  user   2395  sim +0.876  co-rated  12


**Findings:**
User 1's neighbors agree with them nearly perfectly, but look at the co-rated counts: five, five, five. A correlation of 0.98 over five shared movies means almost nothing, and this is exactly the first documented weakness of the algorithm. The minimum co-rating threshold keeps pairs who share fewer than five movies out of the model, but five is still a very small sample to judge taste on.

## The skew problem in numbers

How widespread is the problem? Across all pairs of users, look at how many items they co-rated and what magnitude of correlation that produces. If the correlation were meaningful, it would stay moderate regardless of sample size. It does not.

In [4]:
binary = (R > 0).astype(np.int8)
common = (binary @ binary.T).toarray()
np.fill_diagonal(common, 0)

# Pearson over co-rated items, replicated from the recommender
centered = R.copy().astype(np.float32)
user_of_row = np.repeat(np.arange(len(user_ids)), np.diff(R.indptr))
centered.data -= (np.asarray(R.sum(axis=1)).ravel() / np.asarray(R.getnnz(axis=1)).ravel())[user_of_row]
bin32 = (R > 0).astype(np.float32)
numer = (centered @ centered.T).toarray()
denom = np.sqrt(((centered.multiply(centered)) @ bin32.T).toarray() * (bin32 @ (centered.multiply(centered)).T).toarray())
np.maximum(denom, 1e-9, out=denom)
sim = np.divide(numer, denom, out=np.zeros_like(numer), where=denom > 1e-9)
np.fill_diagonal(sim, 0)

ri, ci = np.triu_indices(len(user_ids), 1)
co = common[ri, ci]
sc = np.abs(sim[ri, ci])

for lo in [2, 5, 10, 20, 50]:
    m = co == lo
    print(f"co-rated = {lo:3d}: {m.sum():>9,} pairs | mean |corr| {sc[m].mean():.3f} | |corr| > 0.8 in {(sc[m] > 0.8).mean():6.2%}")
m = co >= 50
print(f"co-rated >= 50: {m.sum():>9,} pairs | mean |corr| {sc[m].mean():.3f} | |corr| > 0.8 in {(sc[m] > 0.8).mean():6.2%}")

co-rated =   2: 1,569,783 pairs | mean |corr| 0.648 | |corr| > 0.8 in 43.12%
co-rated =   5:   991,442 pairs | mean |corr| 0.407 | |corr| > 0.8 in  8.66%
co-rated =  10:   512,918 pairs | mean |corr| 0.307 | |corr| > 0.8 in  1.44%
co-rated =  20:   196,103 pairs | mean |corr| 0.247 | |corr| > 0.8 in  0.10%
co-rated =  50:    39,282 pairs | mean |corr| 0.215 | |corr| > 0.8 in  0.00%
co-rated >= 50: 1,082,746 pairs | mean |corr| 0.217 | |corr| > 0.8 in  0.00%


**Findings:**
Pairs who shared two movies average |0.65| correlation, and 43% of them are above 0.8. Pairs who shared fifty movies average |0.22|, and essentially none reach 0.8. The correlation shrinks as the sample grows: high correlations on few co-rated items are mostly noise. A model that trusts them is trusting coin flips.

## How many neighbors?

Given the noise, the number of neighbors used matters enormously. A single spurious near-perfect match should not steer the list, but a large crowd of mildly similar users drowns out the few genuinely similar ones. This is a tuning sweep, same split, same evaluation, over neighborhood size.

In [5]:
cols = ["precision@10", "recall@10", "map@10", "ndcg@10", "hit_rate@10"]
rows_sweep = []
for k in [1, 3, 5, 10, 20, 50]:
    m = UserBasedRecommender(k_neighbors=k, min_co_ratings=5).fit(train)
    r = evaluate_ranking(m, train, test, k=10)
    rmse = evaluate_predictions(m, test)["rmse"]
    row = {"k": k, **{c: r[c] for c in cols}, "rmse": rmse}
    rows_sweep.append(row)
sweep = pd.DataFrame(rows_sweep)
print(sweep.round(4).to_string(index=False))

 k  precision@10  recall@10  map@10  ndcg@10  hit_rate@10   rmse
 1        0.0517     0.0218  0.0238   0.0291       0.3272 1.1530
 3        0.0476     0.0170  0.0206   0.0237       0.3056 1.1641
 5        0.0429     0.0143  0.0184   0.0205       0.2834 1.1710
10        0.0367     0.0103  0.0146   0.0151       0.2430 1.1827
20        0.0263     0.0065  0.0099   0.0098       0.1912 1.1904
50        0.0185     0.0038  0.0067   0.0061       0.1377 1.1835


**Findings:**
The best lists use the fewest neighbors. Precision, recall, MAP, NDCG, and hit rate all fall monotonically as k grows from 1 to 50. With noisy similarity, each additional neighbor is more likely to be a spurious match than a genuine one, so the model degrades gracefully but steadily. The default is set to k=5 as a middle ground: nearly the best of the sweep and far more stable than trusting a single correlation.

## Population-wide agreement

The second documented weakness: two users who both like a universally loved movie look similar, but that shared like carries no information about them as individuals. The fix from the literature is to weight items inversely to their popularity when computing the correlation, so agreement on a blockbuster counts less and agreement on an obscure film counts more. The recommender supports this as `weighting="popularity"`. Earlier we saw the two variants already pick completely different neighborhoods; here is what that costs or buys on the metrics.

In [6]:
rows_w = []
for w in ["none", "popularity"]:
    m = UserBasedRecommender(k_neighbors=5, min_co_ratings=5, weighting=w).fit(train)
    r = evaluate_ranking(m, train, test, k=10)
    rmse = evaluate_predictions(m, test)["rmse"]
    rows_w.append({"weighting": w, **{c: r[c] for c in cols}, "rmse": rmse})
weight_cmp = pd.DataFrame(rows_w)
print(weight_cmp.round(4).to_string(index=False))

 weighting  precision@10  recall@10  map@10  ndcg@10  hit_rate@10   rmse
      none        0.0429     0.0143  0.0184   0.0205       0.2834 1.1710
popularity        0.0413     0.0131  0.0170   0.0187       0.2733 1.1784


**Findings:**
Popularity weighting changes the neighborhoods entirely but does not help on this data: every ranking metric is slightly worse. The reason is the co-rating floor. Once pairs must share at least five movies, the worst spurious matches are already gone, and down-weighting blockbusters also discards genuine signal. The fix is principled, and it matters in sparser datasets; on MovieLens-1M with this split it is not a win.

## User-based vs the baselines

Same split, same k=10, all three models against each other.

In [7]:
popularity = PopularityRecommender().fit(train)
content = ContentBasedRecommender().fit(train, items=movies)

comparison = pd.DataFrame({
    "popularity": evaluate_ranking(popularity, train, test, k=10),
    "content-based": evaluate_ranking(content, train, test, k=10),
    "user-based": evaluate_ranking(ubcf, train, test, k=10),
}).T
print(comparison[cols].round(4).to_string())

pred_cols = {
    "model": ["popularity", "content-based", "user-based"],
    "rmse": [
        evaluate_predictions(popularity, test)["rmse"],
        evaluate_predictions(content, test)["rmse"],
        evaluate_predictions(ubcf, test)["rmse"],
    ],
    "mae": [
        evaluate_predictions(popularity, test)["mae"],
        evaluate_predictions(content, test)["mae"],
        evaluate_predictions(ubcf, test)["mae"],
    ],
}
print()
print(pd.DataFrame(pred_cols).round(4).to_string(index=False))

               precision@10  recall@10  map@10  ndcg@10  hit_rate@10
popularity           0.1040     0.0379  0.0565   0.0530       0.4548
content-based        0.0248     0.0116  0.0103   0.0129       0.1680
user-based           0.0429     0.0143  0.0184   0.0205       0.2834

        model   rmse    mae
   popularity 1.1464 0.9439
content-based 1.1464 0.9439
   user-based 1.1710 0.9515


**Findings:**
User-based collaborative filtering beats content-based on every ranking metric, more than doubling precision (0.043 vs 0.025) and hit rate (0.283 vs 0.168) while keeping a lower RMSE. It still trails popularity on every metric. That is the honest state of the art after three stages: a popularity baseline is hard to beat on future ratings, because the test set is dominated by the same blockbusters. User-based CF is the first model to improve on content-based, and it did so using no content at all. RMSE hovers near the global-mean floor, as established in the popularity stage: rating error and ranking quality are different questions.

## It does personalize

The aggregate hides the point of collaborative filtering: the lists are built from each user's own taste. User 1's neighbors are heavy Comedy/Drama/War watchers; user 42's are Comedy/Action watchers, and their top-10s share nothing.

In [8]:
def show_titles(user_id, n=10):
    rec = ubcf.recommend(user_id, k=n)
    titled = pd.DataFrame({"movie_id": rec}).merge(
        movies[["movie_id", "title", "genres"]], on="movie_id"
    )
    print(f"user {user_id}")
    print(titled[["title", "genres"]].to_string(index=False))
    print()

show_titles(1)
show_titles(42)
overlap = set(ubcf.recommend(1, k=10)) & set(ubcf.recommend(42, k=10))
print(f"shared top-10 between users 1 and 42: {len(overlap)} of 10")

user 1
                                 title                 genres
Monty Python and the Holy Grail (1974)               [Comedy]
               Deer Hunter, The (1978)           [Drama, War]
   History of the World: Part I (1981)               [Comedy]
                         Friday (1995)               [Comedy]
                     GoodFellas (1990)         [Crime, Drama]
                   Pulp Fiction (1994)         [Crime, Drama]
                 Reservoir Dogs (1992)      [Crime, Thriller]
                   Patriot, The (2000)   [Action, Drama, War]
              Untouchables, The (1987) [Action, Crime, Drama]
                      Gladiator (2000)        [Action, Drama]

user 42
                                             title                         genres
                        Beverly Hills Ninja (1997)               [Action, Comedy]
                      Three Musketeers, The (1993)    [Action, Adventure, Comedy]
Austin Powers: International Man of Mystery (1997)      

**Findings:**
User 1 gets dark comedy and crime, user 42 gets comedy and action-comedy, and the lists share zero movies. The neighbors' taste is doing the work. This is personalization without a single genre feature in the model.

## New-user cold start

The mirror image of content-based's new-item strength. A user with no ratings has no neighbors, and a user who appears only in the test set never made it into the matrix at all. Collaborative filtering needs the user's own history to find similar people, so the model falls back to the global mean and returns no list.

In [9]:
print(f"train users: {len(user_ids):,} | users with zero valid neighbors: {(ubcf._neighbor_sim.max(axis=1) == 0).sum()}")

# a user id that does not exist in the training split at all
print(f"recommend for a brand-new user: {ubcf.recommend(999999, k=5)}")
print(f"predict for a brand-new user:   {ubcf.predict(999999, 1):.4f}  (global mean)")

train users: 6,040 | users with zero valid neighbors: 4
recommend for a brand-new user: []
predict for a brand-new user:   3.6169  (global mean)


**Findings:**
Four real train users have no valid neighbors and get nothing; a brand-new user gets the empty list and the global mean. Content-based covered new items but needed genre tags; user-based covers taste but needs history. Neither covers everything, and the clean way out, seen in later stages, is to combine signals.